# Preprocessing v2.0 — Semua Skenario (A-Manual, A-Gabungan, B-Gabungan)

## Desain Splitting
| Skenario | Train | Test | Sumber Test |
|---|---|---|---|
| A-Manual | 80% manual | 20% manual (~3.194) | Manual murni |
| A-Gabungan | 80% gabungan | 20% gabungan (~11.839) | Gabungan 20% |
| B-Gabungan | 80% gabungan | **3.194 dari gabungan** | Sample gabungan |

## Eksperimen (9 total)
```
LR      : A-Manual, A-Gabungan, B-Gabungan
XGB NoTune: A-Manual, A-Gabungan, B-Gabungan
XGB Tuned : A-Manual (~22 mnt), A-Gabungan (~90 mnt), B-Gabungan (~90 mnt)
```

## Alur Notebook
```
Cell 1  → Setup & konfigurasi
Cell 2  → Splitting (3 skenario)
Cell 3  → Fungsi TF-IDF Bigram + Evaluasi
Cell 4  → LR (A-Manual, A-Gabungan, B-Gabungan)
Cell 5  → XGB NoTune (A-Manual, A-Gabungan, B-Gabungan)
Cell 6a → XGB Tuned A-Manual (~22 menit)
Cell 6b → XGB Tuned A-Gabungan (~90 menit)
Cell 6c → XGB Tuned B-Gabungan (~90 menit)
Cell 7  → Perbandingan lengkap v1.0 vs v2.0
```


## Cell 1 — Setup & Konfigurasi

In [1]:
import pandas as pd
import numpy as np
import os, pickle, time, warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report
)

# ── PATH ─────────────────────────────────────────────────────────────────────
BASE_DIR   = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
PREP_DIR   = os.path.join(BASE_DIR, 'prep_v2')
SPLIT_DIR  = os.path.join(BASE_DIR, 'splits_v2b')
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
for d in [SPLIT_DIR, MODEL_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

SEED        = 42
LABEL_MAP   = {'keluhan': 0, 'saran': 1, 'pujian': 2}
CLASS_NAMES = ['keluhan', 'saran', 'pujian']

# k_best optimal (dari grid search k sebelumnya)
K_MANUAL   = 2000
K_GABUNGAN = 15000

# ── Hasil v1.0 untuk perbandingan ─────────────────────────────────────────────
HASIL_V1 = {
    # LR
    'LR-A-Manual'         : {'acc':79.13,'f1':0.7582,'f1_k':0.8499,'f1_s':0.5893,'f1_p':0.8355},
    'LR-A-Gabungan'       : {'acc':81.20,'f1':0.7675,'f1_k':0.8684,'f1_s':0.5941,'f1_p':0.8401},
    'LR-B-Gabungan'       : {'acc':80.96,'f1':0.7677,'f1_k':0.8677,'f1_s':0.5998,'f1_p':0.8354},
    # XGB NoTune
    'XGB-NoTune-A-Manual'   : {'acc':77.86,'f1':0.7473,'f1_k':0.8416,'f1_s':0.5727,'f1_p':0.8275},
    'XGB-NoTune-A-Gabungan' : {'acc':78.80,'f1':0.7458,'f1_k':0.8521,'f1_s':0.5510,'f1_p':0.8344},
    'XGB-NoTune-B-Gabungan' : {'acc':78.29,'f1':0.7417,'f1_k':0.8489,'f1_s':0.5474,'f1_p':0.8290},
    # XGB Tuned
    'XGB-Tuned-A-Manual'    : {'acc':79.49,'f1':0.7574,'f1_k':0.8541,'f1_s':0.5880,'f1_p':0.8301},
    'XGB-Tuned-A-Gabungan'  : {'acc':82.27,'f1':0.7750,'f1_k':0.8793,'f1_s':0.5923,'f1_p':0.8532},
    'XGB-Tuned-B-Gabungan'  : {'acc':81.73,'f1':0.7685,'f1_k':0.8774,'f1_s':0.5837,'f1_p':0.8444},
}

# Cek file preprocessed
path_full = os.path.join(PREP_DIR, 'data_preprocessed_v2.csv')
if os.path.exists(path_full):
    _df = pd.read_csv(path_full)
    print(f'✅ data_preprocessed_v2.csv: {len(_df):,} baris')
    del _df
else:
    print('❌ File tidak ditemukan! Jalankan notebook preprocessing_v2_final dulu.')
print('\n✅ Setup selesai!')

✅ data_preprocessed_v2.csv: 59,864 baris

✅ Setup selesai!


## Cell 2 — Splitting (3 Skenario)

```
A-Manual   : 80% manual train  | 20% manual test (~3.194)
A-Gabungan : 80% gab train     | 20% gab test  (~11.839)
B-Gabungan : 80% gab train     | 3.194 baris dari gabungan sebagai test
```


In [2]:
# Load data preprocessed v2.0
print('Loading data_preprocessed_v2.csv...')
df = pd.read_csv(os.path.join(PREP_DIR, 'data_preprocessed_v2.csv'))
df['label_enc'] = df['label_pks'].map(LABEL_MAP)
print(f'Total: {len(df):,} baris')

# Pisah manual vs otomatis
df_manual = df[df['confidence'] == 1.0].copy().reset_index(drop=True)
df_auto   = df[df['confidence'] <  1.0].copy().reset_index(drop=True)
df_gab    = df.copy().reset_index(drop=True)

print(f'  Manual (conf=1.0)   : {len(df_manual):,}')
print(f'  Otomatis (conf<1.0) : {len(df_auto):,}')
print(f'  Gabungan (total)    : {len(df_gab):,}')

# ── A-Manual: 80:20 dari data manual ─────────────────────────────────────────
man_train, man_test = train_test_split(
    df_manual, test_size=0.2, random_state=SEED,
    stratify=df_manual['label_enc']
)

# ── A-Gabungan: 80:20 dari data GABUNGAN ──────────────────────────────────────
agab_train, agab_test = train_test_split(
    df_gab, test_size=0.2, random_state=SEED,
    stratify=df_gab['label_enc']
)

# ── B-Gabungan: train=80% gabungan | test=3.194 dari GABUNGAN ────────────────
# test_size = jumlah baris yang sama dengan man_test (bukan manual murni)
bgab_train, bgab_test = train_test_split(
    df_gab,
    test_size=len(man_test),   # ← 3.194 baris dari gabungan
    random_state=SEED,
    stratify=df_gab['label_enc']
)

# ── Ringkasan ────────────────────────────────────────────────────────────────
print(f'\n{"="*65}')
print(f'{"Skenario":15s} {"N Train":>10s} {"N Test":>10s} {"Sumber Test"}')
print(f'{"─"*65}')
print(f'{"A-Manual":15s} {len(man_train):>10,} {len(man_test):>10,}  20% manual murni')
print(f'{"A-Gabungan":15s} {len(agab_train):>10,} {len(agab_test):>10,}  20% gabungan')
print(f'{"B-Gabungan":15s} {len(bgab_train):>10,} {len(bgab_test):>10,}  {len(man_test)} baris dari gabungan')
print(f'{"="*65}')

# Validasi B-Gabungan
n_man_in_b = (bgab_test['confidence'] == 1.0).sum()
n_aut_in_b = (bgab_test['confidence'] <  1.0).sum()
print(f'\nKomposisi test B-Gabungan ({len(bgab_test):,} baris):')
print(f'  Manual (conf=1.0)   : {n_man_in_b:,} ({n_man_in_b/len(bgab_test)*100:.1f}%)')
print(f'  Otomatis (conf<1.0) : {n_aut_in_b:,}  ({n_aut_in_b/len(bgab_test)*100:.1f}%)')
print(f'\nDistribusi label test B-Gabungan:')
for lbl, cnt in bgab_test['label_pks'].value_counts().items():
    print(f'  {lbl:10s}: {cnt:5,} ({cnt/len(bgab_test)*100:.1f}%)')

# Simpan splits
splits = {
    'v2b_manual_train.csv'  : man_train,
    'v2b_manual_test.csv'   : man_test,
    'v2b_agab_train.csv'    : agab_train,
    'v2b_agab_test.csv'     : agab_test,
    'v2b_bgab_train.csv'    : bgab_train,
    'v2b_bgab_test.csv'     : bgab_test,
}
print(f'\nMenyimpan splits ke {SPLIT_DIR}...')
for fname, df_sp in splits.items():
    path = os.path.join(SPLIT_DIR, fname)
    df_sp.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'  ✅ {fname}: {len(df_sp):,}')

# Simpan konfigurasi skenario untuk cell berikutnya
SKENARIO = [
    {'nama':'A-Manual',   'tr':'v2b_manual_train.csv', 'te':'v2b_manual_test.csv', 'k':K_MANUAL},
    {'nama':'A-Gabungan', 'tr':'v2b_agab_train.csv',   'te':'v2b_agab_test.csv',   'k':K_GABUNGAN},
    {'nama':'B-Gabungan', 'tr':'v2b_bgab_train.csv',   'te':'v2b_bgab_test.csv',   'k':K_GABUNGAN},
]
print('\n✅ Splitting selesai!')

Loading data_preprocessed_v2.csv...
Total: 59,864 baris
  Manual (conf=1.0)   : 16,343
  Otomatis (conf<1.0) : 43,521
  Gabungan (total)    : 59,864

Skenario           N Train     N Test Sumber Test
─────────────────────────────────────────────────────────────────
A-Manual            13,074      3,269  20% manual murni
A-Gabungan          47,891     11,973  20% gabungan
B-Gabungan          56,595      3,269  3269 baris dari gabungan

Komposisi test B-Gabungan (3,269 baris):
  Manual (conf=1.0)   : 900 (27.5%)
  Otomatis (conf<1.0) : 2,369  (72.5%)

Distribusi label test B-Gabungan:
  keluhan   : 2,206 (67.5%)
  pujian    :   614 (18.8%)
  saran     :   449 (13.7%)

Menyimpan splits ke C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\splits_v2b...
  ✅ v2b_manual_train.csv: 13,074
  ✅ v2b_manual_test.csv: 3,269
  ✅ v2b_agab_train.csv: 47,891
  ✅ v2b_agab_test.csv: 11,973
  ✅ v2b_bgab_train.csv: 56,595
  ✅ v2b_bgab_test.csv: 3,269

✅ Splitting selesai!


## Cell 3 — Fungsi TF-IDF Bigram + Evaluasi

In [3]:
def build_features(train_texts, train_labels, k_best, max_features=100000):
    """TF-IDF Bigram (1,2) + Chi-Square — v2.0."""
    tfidf = TfidfVectorizer(
        ngram_range   = (1, 2),
        max_features  = max_features,
        sublinear_tf  = True,
        min_df        = 2,
        token_pattern = r'[a-zA-Z_][a-zA-Z_]+',
    )
    X     = tfidf.fit_transform(train_texts)
    y_enc = np.array([LABEL_MAP[l] for l in train_labels])
    sel   = SelectKBest(chi2, k=min(k_best, X.shape[1]))
    X_sel = sel.fit_transform(X, y_enc)

    feat_names   = np.array(tfidf.get_feature_names_out())[sel.get_support()]
    negasi_feats = [f for f in feat_names if '_' in f and '__' not in f]
    bigram_feats = [f for f in feat_names if ' ' in f]
    print(f'  TF-IDF: {X.shape[1]:,} → Chi2 k={k_best:,} → {X_sel.shape[1]:,} fitur')
    print(f'  Token tidak_X: {len(negasi_feats):,} | Bigram: {len(bigram_feats):,}')
    return tfidf, sel, X_sel, y_enc


def get_xy(skenario, split_dir):
    """Load data dan encode label."""
    df_tr = pd.read_csv(os.path.join(split_dir, skenario['tr']))
    df_te = pd.read_csv(os.path.join(split_dir, skenario['te']))
    X_tr  = df_tr['text_v2'].fillna('').values
    y_tr  = df_tr['label_pks'].values
    X_te  = df_te['text_v2'].fillna('').values
    y_te  = np.array([LABEL_MAP[l] for l in df_te['label_pks'].values])
    print(f'  Train: {len(df_tr):,} | Test: {len(df_te):,}')
    return X_tr, y_tr, X_te, y_te


def get_sw(y_train):
    sw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    return np.array([sw[y] for y in y_train])


def evaluate(y_test, y_pred, nama, v1_key=None):
    """Evaluasi model dan tampilkan hasil + perbandingan v1.0."""
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1s  = f1_score(y_test, y_pred, average=None, labels=[0,1,2], zero_division=0)
    prec = precision_score(y_test, y_pred, average=None, labels=[0,1,2], zero_division=0)
    rec  = recall_score(y_test, y_pred, average=None, labels=[0,1,2], zero_division=0)

    print(f'\n{"="*60}')
    print(f'HASIL — {nama}')
    print(f'{"="*60}')
    print(f'  Accuracy   : {acc*100:.2f}%')
    print(f'  Macro F1   : {f1:.4f}')
    print(f'  F1 Keluhan : {f1s[0]:.4f}  Prec:{prec[0]:.4f}  Rec:{rec[0]:.4f}')
    print(f'  F1 Saran   : {f1s[1]:.4f}  Prec:{prec[1]:.4f}  Rec:{rec[1]:.4f}')
    print(f'  F1 Pujian  : {f1s[2]:.4f}  Prec:{prec[2]:.4f}  Rec:{rec[2]:.4f}')
    print()
    print(classification_report(y_test, y_pred,
          target_names=CLASS_NAMES, digits=4, zero_division=0))

    if v1_key and v1_key in HASIL_V1:
        v1 = HASIL_V1[v1_key]
        print(f'  Perbandingan vs v1.0 ({v1_key}):')
        for label, k_v1, v2_val in [
            ('Accuracy', 'acc',  acc*100),
            ('Macro F1', 'f1',   f1),
            ('F1 Saran', 'f1_s', f1s[1]),
        ]:
            delta = v2_val - v1[k_v1]
            ikon  = '✅↑' if delta>0.001 else ('❌↓' if delta<-0.001 else '→')
            print(f'    {label:10s}: {v1[k_v1]:.4f} → {v2_val:.4f}  {ikon} {delta:+.4f}')

    return {'nama':nama,'acc':acc,'f1':f1,'f1_k':f1s[0],'f1_s':f1s[1],'f1_p':f1s[2]}


def save_model(model, vec_dict, nama):
    key = nama.replace('-','_').replace(' ','_')
    pickle.dump(model, open(os.path.join(MODEL_DIR, f'{key}.pkl'), 'wb'))
    pickle.dump(vec_dict, open(os.path.join(MODEL_DIR, f'vec_{key}.pkl'), 'wb'))
    print(f'  ✅ Model tersimpan: {key}.pkl')


print('✅ Fungsi siap!')
print(f'\nSkenario yang akan diuji:')
for s in SKENARIO:
    print(f'  {s["nama"]:12s} k={s["k"]:,}')

✅ Fungsi siap!

Skenario yang akan diuji:
  A-Manual     k=2,000
  A-Gabungan   k=15,000
  B-Gabungan   k=15,000


## Cell 4 — Logistic Regression (A-Manual, A-Gabungan, B-Gabungan)

In [4]:
hasil_lr = []

for s in SKENARIO:
    nama   = f'LR-v2-{s["nama"]}'
    v1_key = f'LR-{s["nama"]}'
    print(f'\n{"─"*60}')
    print(f'SKENARIO: {nama} (k={s["k"]:,})')
    print(f'{"─"*60}')

    X_tr, y_tr, X_te, y_te = get_xy(s, SPLIT_DIR)
    tfidf, sel, X_train, y_train = build_features(X_tr, y_tr, s['k'])
    X_test = sel.transform(tfidf.transform(X_te))

    sw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    lr = LogisticRegression(
        multi_class='multinomial', solver='lbfgs', C=1.0,
        max_iter=1000, class_weight=dict(enumerate(sw)),
        random_state=SEED, n_jobs=-1,
    )
    t0 = time.time()
    lr.fit(X_train, y_train)
    print(f'  Training LR: {time.time()-t0:.2f}s')

    hasil = evaluate(y_te, lr.predict(X_test), nama, v1_key)
    hasil_lr.append(hasil)
    save_model(lr, {'tfidf':tfidf,'selector':sel}, nama)

print('\n✅ LR selesai!')


────────────────────────────────────────────────────────────
SKENARIO: LR-v2-A-Manual (k=2,000)
────────────────────────────────────────────────────────────
  Train: 13,074 | Test: 3,269
  TF-IDF: 14,262 → Chi2 k=2,000 → 2,000 fitur
  Token tidak_X: 166 | Bigram: 1,170
  Training LR: 2.49s

HASIL — LR-v2-A-Manual
  Accuracy   : 81.40%
  Macro F1   : 0.7804
  F1 Keluhan : 0.8615  Prec:0.9021  Rec:0.8245
  F1 Saran   : 0.6218  Prec:0.5483  Rec:0.7179
  F1 Pujian  : 0.8580  Prec:0.8534  Rec:0.8627

              precision    recall  f1-score   support

     keluhan     0.9021    0.8245    0.8615      2068
       saran     0.5483    0.7179    0.6218       553
      pujian     0.8534    0.8627    0.8580       648

    accuracy                         0.8140      3269
   macro avg     0.7680    0.8017    0.7804      3269
weighted avg     0.8326    0.8140    0.8203      3269

  Perbandingan vs v1.0 (LR-A-Manual):
    Accuracy  : 79.1300 → 81.4010  ✅↑ +2.2710
    Macro F1  : 0.7582 → 0.7804  

## Cell 5 — XGBoost NoTune (A-Manual, A-Gabungan, B-Gabungan)

In [5]:
hasil_notune = []

for s in SKENARIO:
    nama   = f'XGB-v2-NoTune-{s["nama"]}'
    v1_key = f'XGB-NoTune-{s["nama"]}'
    print(f'\n{"─"*60}')
    print(f'SKENARIO: {nama} (k={s["k"]:,})')
    print(f'{"─"*60}')

    X_tr, y_tr, X_te, y_te = get_xy(s, SPLIT_DIR)
    tfidf, sel, X_train, y_train = build_features(X_tr, y_tr, s['k'])
    X_test = sel.transform(tfidf.transform(X_te))
    sw_arr = get_sw(y_train)

    model = xgb.XGBClassifier(
        objective='multi:softmax', num_class=3,
        eval_metric='mlogloss',
        n_estimators=300, max_depth=6,
        learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8,
        device='cpu', random_state=SEED,
        n_jobs=-1, use_label_encoder=False,
    )
    t0 = time.time()
    model.fit(X_train, y_train, sample_weight=sw_arr,
              eval_set=[(X_test, y_te)], verbose=False)
    print(f'  Training: {time.time()-t0:.1f}s')

    hasil = evaluate(y_te, model.predict(X_test), nama, v1_key)
    hasil_notune.append(hasil)
    save_model(model, {'tfidf':tfidf,'selector':sel}, nama)

print('\n✅ XGBoost NoTune selesai!')


────────────────────────────────────────────────────────────
SKENARIO: XGB-v2-NoTune-A-Manual (k=2,000)
────────────────────────────────────────────────────────────
  Train: 13,074 | Test: 3,269
  TF-IDF: 14,262 → Chi2 k=2,000 → 2,000 fitur
  Token tidak_X: 166 | Bigram: 1,170
  Training: 8.4s

HASIL — XGB-v2-NoTune-A-Manual
  Accuracy   : 79.75%
  Macro F1   : 0.7632
  F1 Keluhan : 0.8523  Prec:0.9128  Rec:0.7993
  F1 Saran   : 0.5876  Prec:0.5000  Rec:0.7125
  F1 Pujian  : 0.8498  Prec:0.8358  Rec:0.8642

              precision    recall  f1-score   support

     keluhan     0.9128    0.7993    0.8523      2068
       saran     0.5000    0.7125    0.5876       553
      pujian     0.8358    0.8642    0.8498       648

    accuracy                         0.7975      3269
   macro avg     0.7495    0.7920    0.7632      3269
weighted avg     0.8277    0.7975    0.8070      3269

  Perbandingan vs v1.0 (XGB-NoTune-A-Manual):
    Accuracy  : 77.8600 → 79.7492  ✅↑ +1.8892
    Macro F1 

## Cell 6a — XGBoost Tuned: A-Manual (~22 menit)
> Jalankan cell ini sendiri, tunggu selesai baru lanjut ke 6b


In [ ]:
PARAM_GRID = {
    'n_estimators'    : [300, 500],
    'max_depth'       : [4, 6, 8],
    'learning_rate'   : [0.05, 0.1, 0.2],
    'subsample'       : [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}
print(f'72 kombinasi × 5 fold = 360 fits')

hasil_tuned = []  # list hasil tuned (diisi bertahap per cell)

def run_tuned(s, hasil_tuned_list):
    nama   = f'XGB-v2-Tuned-{s["nama"]}'
    v1_key = f'XGB-Tuned-{s["nama"]}'
    print(f'\n{"="*60}')
    print(f'SKENARIO: {nama} (k={s["k"]:,})')
    print(f'{"="*60}')

    X_tr, y_tr, X_te, y_te = get_xy(s, SPLIT_DIR)
    tfidf, sel, X_train, y_train = build_features(X_tr, y_tr, s['k'])
    X_test = sel.transform(tfidf.transform(X_te))
    sw_arr = get_sw(y_train)

    base_xgb = xgb.XGBClassifier(
        objective='multi:softmax', num_class=3,
        eval_metric='mlogloss', device='cpu',
        random_state=SEED, n_jobs=1,
        use_label_encoder=False,
    )
    cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    grid = GridSearchCV(
        base_xgb, PARAM_GRID, cv=cv,
        scoring='f1_macro', n_jobs=-1,
        verbose=1, refit=False,
    )
    t0 = time.time()
    grid.fit(X_train, y_train, sample_weight=sw_arr)
    print(f'\nSelesai: {(time.time()-t0)/60:.1f} menit')
    print(f'Best params : {grid.best_params_}')
    print(f'Best CV F1  : {grid.best_score_:.4f}')

    pd.DataFrame(grid.cv_results_).to_csv(
        os.path.join(RESULT_DIR, f'cv_{nama.replace("-","_")}.csv'), index=False)

    best = xgb.XGBClassifier(
        objective='multi:softmax', num_class=3,
        eval_metric='mlogloss', device='cpu',
        random_state=SEED, n_jobs=-1,
        use_label_encoder=False, **grid.best_params_
    )
    t0 = time.time()
    best.fit(X_train, y_train, sample_weight=sw_arr,
             eval_set=[(X_test, y_te)], verbose=False)
    print(f'Retrain: CPU ✅ ({time.time()-t0:.1f}s)')

    hasil = evaluate(y_te, best.predict(X_test), nama, v1_key)
    hasil_tuned_list.append(hasil)
    save_model(best, {'tfidf':tfidf,'selector':sel}, nama)
    return hasil_tuned_list


In [ ]:
# Jalankan A-Manual
s_manual = SKENARIO[0]  # A-Manual
hasil_tuned = run_tuned(s_manual, hasil_tuned)
print('\n✅ XGB Tuned A-Manual selesai! Lanjut ke Cell 6b.')

72 kombinasi × 5 fold = 360 fits

SKENARIO: XGB-v2-Tuned-A-Manual (k=2,000)
  Train: 13,074 | Test: 3,269
  TF-IDF: 14,262 → Chi2 k=2,000 → 2,000 fitur
  Token tidak_X: 166 | Bigram: 1,170
Fitting 5 folds for each of 72 candidates, totalling 360 fits

Selesai: 10.7 menit
Best params : {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 4, 'n_estimators': 500, 'subsample': 1.0}
Best CV F1  : 0.7915
Retrain: CPU ✅ (10.9s)

HASIL — XGB-v2-Tuned-A-Manual
  Accuracy   : 80.79%
  Macro F1   : 0.7721
  F1 Keluhan : 0.8614  Prec:0.9124  Rec:0.8158
  F1 Saran   : 0.5988  Prec:0.5193  Rec:0.7071
  F1 Pujian  : 0.8563  Prec:0.8441  Rec:0.8688

              precision    recall  f1-score   support

     keluhan     0.9124    0.8158    0.8614      2068
       saran     0.5193    0.7071    0.5988       553
      pujian     0.8441    0.8688    0.8563       648

    accuracy                         0.8079      3269
   macro avg     0.7586    0.7972    0.7721      3269
weighted avg     0.8323 

## Cell 6b — XGBoost Tuned: A-Gabungan (~90 menit)
> Pastikan Cell 6a sudah selesai sebelum menjalankan ini


In [7]:
# Jalankan A-Gabungan
s_agab = SKENARIO[1]  # A-Gabungan
hasil_tuned = run_tuned(s_agab, hasil_tuned)
print('\n✅ XGB Tuned A-Gabungan selesai! Lanjut ke Cell 6c.')


SKENARIO: XGB-v2-Tuned-A-Gabungan (k=15,000)
  Train: 47,891 | Test: 11,973
  TF-IDF: 41,505 → Chi2 k=15,000 → 15,000 fitur
  Token tidak_X: 1,753 | Bigram: 11,469
Fitting 5 folds for each of 72 candidates, totalling 360 fits

Selesai: 67.0 menit
Best params : {'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 6, 'n_estimators': 500, 'subsample': 1.0}
Best CV F1  : 0.8413
Retrain: CPU ✅ (197.6s)

HASIL — XGB-v2-Tuned-A-Gabungan
  Accuracy   : 84.48%
  Macro F1   : 0.8037
  F1 Keluhan : 0.8893  Prec:0.9538  Rec:0.8329
  F1 Saran   : 0.6506  Prec:0.5425  Rec:0.8125
  F1 Pujian  : 0.8712  Prec:0.8347  Rec:0.9111

              precision    recall  f1-score   support

     keluhan     0.9538    0.8329    0.8893      8080
       saran     0.5425    0.8125    0.6506      1643
      pujian     0.8347    0.9111    0.8712      2250

    accuracy                         0.8448     11973
   macro avg     0.7770    0.8522    0.8037     11973
weighted avg     0.8750    0.8448    0.8531  

## Cell 6c — XGBoost Tuned: B-Gabungan (~90 menit)
> Pastikan Cell 6b sudah selesai sebelum menjalankan ini


In [5]:
# ── JALANKAN DI RUMAH — Load hasil sebelumnya dulu ────────────────────────
import pickle, os, pandas as pd, numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# Load semua hasil model yang sudah tersimpan
def load_eval(nama_model, vec_nama, split_te, v1_key=None):
    """Load model tersimpan dan evaluasi ulang pada test set."""
    model = pickle.load(open(os.path.join(MODEL_DIR, f'{nama_model}.pkl'), 'rb'))
    vec   = pickle.load(open(os.path.join(MODEL_DIR, f'vec_{nama_model}.pkl'), 'rb'))
    tfidf = vec['tfidf']
    sel   = vec['selector']

    df_te  = pd.read_csv(os.path.join(SPLIT_DIR, split_te))
    X_te   = df_te['text_v2'].fillna('').values
    y_te   = np.array([LABEL_MAP[l] for l in df_te['label_pks'].values])
    X_test = sel.transform(tfidf.transform(X_te))
    y_pred = model.predict(X_test)

    return evaluate(y_te, y_pred, nama_model.replace('_','-'), v1_key)

# Rebuild hasil_lr, hasil_notune, hasil_tuned dari model tersimpan
print('Loading hasil dari model tersimpan...')
hasil_lr = [
    load_eval('LR_v2_A_Manual',   'LR_v2_A_Manual',
              'v2b_manual_test.csv',   'LR-A-Manual'),
    load_eval('LR_v2_A_Gabungan', 'LR_v2_A_Gabungan',
              'v2b_agab_test.csv',     'LR-A-Gabungan'),
    load_eval('LR_v2_B_Gabungan', 'LR_v2_B_Gabungan',
              'v2b_bgab_test.csv',     'LR-B-Gabungan'),
]
hasil_notune = [
    load_eval('XGB_v2_NoTune_A_Manual',   'XGB_v2_NoTune_A_Manual',
              'v2b_manual_test.csv',        'XGB-NoTune-A-Manual'),
    load_eval('XGB_v2_NoTune_A_Gabungan', 'XGB_v2_NoTune_A_Gabungan',
              'v2b_agab_test.csv',          'XGB-NoTune-A-Gabungan'),
    load_eval('XGB_v2_NoTune_B_Gabungan', 'XGB_v2_NoTune_B_Gabungan',
              'v2b_bgab_test.csv',          'XGB-NoTune-B-Gabungan'),
]
hasil_tuned = [
    load_eval('XGB_v2_Tuned_A_Manual',   'XGB_v2_Tuned_A_Manual',
              'v2b_manual_test.csv',       'XGB-Tuned-A-Manual'),
    load_eval('XGB_v2_Tuned_A_Gabungan', 'XGB_v2_Tuned_A_Gabungan',
              'v2b_agab_test.csv',         'XGB-Tuned-A-Gabungan'),
]
print('✅ Semua hasil berhasil di-reload!')
print(f'  LR     : {len(hasil_lr)} skenario')
print(f'  NoTune : {len(hasil_notune)} skenario')
print(f'  Tuned  : {len(hasil_tuned)} skenario (B-Gabungan belum)')

Loading hasil dari model tersimpan...

HASIL — LR-v2-A-Manual
  Accuracy   : 81.40%
  Macro F1   : 0.7804
  F1 Keluhan : 0.8615  Prec:0.9021  Rec:0.8245
  F1 Saran   : 0.6218  Prec:0.5483  Rec:0.7179
  F1 Pujian  : 0.8580  Prec:0.8534  Rec:0.8627

              precision    recall  f1-score   support

     keluhan     0.9021    0.8245    0.8615      2068
       saran     0.5483    0.7179    0.6218       553
      pujian     0.8534    0.8627    0.8580       648

    accuracy                         0.8140      3269
   macro avg     0.7680    0.8017    0.7804      3269
weighted avg     0.8326    0.8140    0.8203      3269

  Perbandingan vs v1.0 (LR-A-Manual):
    Accuracy  : 79.1300 → 81.4010  ✅↑ +2.2710
    Macro F1  : 0.7582 → 0.7804  ✅↑ +0.0222
    F1 Saran  : 0.5893 → 0.6218  ✅↑ +0.0325

HASIL — LR-v2-A-Gabungan
  Accuracy   : 86.16%
  Macro F1   : 0.8212
  F1 Keluhan : 0.9015  Prec:0.9505  Rec:0.8573
  F1 Saran   : 0.6800  Prec:0.5872  Rec:0.8077
  F1 Pujian  : 0.8821  Prec:0.8503 

In [10]:
# Jalankan B-Gabungan
s_bgab = SKENARIO[2]  # B-Gabungan
hasil_tuned = run_tuned(s_bgab, hasil_tuned)
print('\n✅ XGB Tuned B-Gabungan selesai! Lanjut ke Cell 7 (Perbandingan).')


SKENARIO: XGB-v2-Tuned-B-Gabungan (k=15,000)
  Train: 56,595 | Test: 3,269
  TF-IDF: 48,063 → Chi2 k=15,000 → 15,000 fitur
  Token tidak_X: 1,696 | Bigram: 11,522
Fitting 5 folds for each of 72 candidates, totalling 360 fits

Selesai: 97.0 menit
Best params : {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 8, 'n_estimators': 500, 'subsample': 1.0}
Best CV F1  : 0.8441
Retrain: CPU ✅ (279.8s)

HASIL — XGB-v2-Tuned-B-Gabungan
  Accuracy   : 84.89%
  Macro F1   : 0.8073
  F1 Keluhan : 0.8935  Prec:0.9509  Rec:0.8427
  F1 Saran   : 0.6613  Prec:0.5542  Rec:0.8196
  F1 Pujian  : 0.8671  Prec:0.8431  Rec:0.8925

              precision    recall  f1-score   support

     keluhan     0.9509    0.8427    0.8935      2206
       saran     0.5542    0.8196    0.6613       449
      pujian     0.8431    0.8925    0.8671       614

    accuracy                         0.8489      3269
   macro avg     0.7827    0.8516    0.8073      3269
weighted avg     0.8762    0.8489    0.8567   

## Cell 7 — Perbandingan Lengkap v1.0 vs v2.0

In [15]:
# ── RELOAD hasil dari model .pkl yang sudah tersimpan ─────────────────────
print('Reloading hasil dari model tersimpan...')

def load_eval(model_key, vec_key, test_file, nama_tampil, v1_key=None):
    """Load model dari .pkl dan evaluasi ulang pada test set."""
    try:
        model = pickle.load(open(
            os.path.join(MODEL_DIR, f'{model_key}.pkl'), 'rb'))
        vec   = pickle.load(open(
            os.path.join(MODEL_DIR, f'vec_{vec_key}.pkl'), 'rb'))

        df_te  = pd.read_csv(os.path.join(SPLIT_DIR, test_file))
        X_te   = df_te['text_v2'].fillna('').values
        y_te   = np.array([LABEL_MAP[l] for l in df_te['label_pks'].values])
        X_test = vec['selector'].transform(
                     vec['tfidf'].transform(X_te))
        y_pred = model.predict(X_test)

        h = evaluate(y_te, y_pred, nama_tampil, v1_key)
        print(f'  ✅ {nama_tampil}')
        return h
    except FileNotFoundError as e:
        print(f'  ❌ File tidak ditemukan: {e}')
        return None

# Rebuild hasil_lr
hasil_lr = []
for nama_model, nama_vec, test_file, nama_tampil, v1_key in [
    ('LR_v2_A_Manual',   'LR_v2_A_Manual',
     'v2b_manual_test.csv', 'LR-v2-A-Manual',   'LR-A-Manual'),
    ('LR_v2_A_Gabungan', 'LR_v2_A_Gabungan',
     'v2b_agab_test.csv',   'LR-v2-A-Gabungan', 'LR-A-Gabungan'),
    ('LR_v2_B_Gabungan', 'LR_v2_B_Gabungan',
     'v2b_bgab_test.csv',   'LR-v2-B-Gabungan', 'LR-B-Gabungan'),
]:
    h = load_eval(nama_model, nama_vec, test_file, nama_tampil, v1_key)
    if h: hasil_lr.append(h)

# Rebuild hasil_notune
hasil_notune = []
for nama_model, nama_vec, test_file, nama_tampil, v1_key in [
    ('XGB_v2_NoTune_A_Manual',   'XGB_v2_NoTune_A_Manual',
     'v2b_manual_test.csv',        'XGB-v2-NoTune-A-Manual',   'XGB-NoTune-A-Manual'),
    ('XGB_v2_NoTune_A_Gabungan', 'XGB_v2_NoTune_A_Gabungan',
     'v2b_agab_test.csv',          'XGB-v2-NoTune-A-Gabungan', 'XGB-NoTune-A-Gabungan'),
    ('XGB_v2_NoTune_B_Gabungan', 'XGB_v2_NoTune_B_Gabungan',
     'v2b_bgab_test.csv',          'XGB-v2-NoTune-B-Gabungan', 'XGB-NoTune-B-Gabungan'),
]:
    h = load_eval(nama_model, nama_vec, test_file, nama_tampil, v1_key)
    if h: hasil_notune.append(h)

# Rebuild hasil_tuned (A-Manual & A-Gabungan dari pkl,
#                      B-Gabungan sudah ada di memory)
hasil_tuned_reload = []
for nama_model, nama_vec, test_file, nama_tampil, v1_key in [
    ('XGB_v2_Tuned_A_Manual',   'XGB_v2_Tuned_A_Manual',
     'v2b_manual_test.csv',       'XGB-v2-Tuned-A-Manual',   'XGB-Tuned-A-Manual'),
    ('XGB_v2_Tuned_A_Gabungan', 'XGB_v2_Tuned_A_Gabungan',
     'v2b_agab_test.csv',         'XGB-v2-Tuned-A-Gabungan', 'XGB-Tuned-A-Gabungan'),
]:
    h = load_eval(nama_model, nama_vec, test_file, nama_tampil, v1_key)
    if h: hasil_tuned_reload.append(h)

# Gabungkan: reload A-Manual & A-Gabungan + B-Gabungan yang sudah ada
hasil_tuned_bgab = [h for h in hasil_tuned
                    if 'B-Gabungan' in h['nama']]
hasil_tuned = hasil_tuned_reload + hasil_tuned_bgab

print(f'\nRingkasan reload:')
print(f'  hasil_lr     : {len(hasil_lr)} ✅')
print(f'  hasil_notune : {len(hasil_notune)} ✅')
print(f'  hasil_tuned  : {len(hasil_tuned)} ✅')
print('\nSekarang jalankan Cell 7!')

Reloading hasil dari model tersimpan...

HASIL — LR-v2-A-Manual
  Accuracy   : 81.40%
  Macro F1   : 0.7804
  F1 Keluhan : 0.8615  Prec:0.9021  Rec:0.8245
  F1 Saran   : 0.6218  Prec:0.5483  Rec:0.7179
  F1 Pujian  : 0.8580  Prec:0.8534  Rec:0.8627

              precision    recall  f1-score   support

     keluhan     0.9021    0.8245    0.8615      2068
       saran     0.5483    0.7179    0.6218       553
      pujian     0.8534    0.8627    0.8580       648

    accuracy                         0.8140      3269
   macro avg     0.7680    0.8017    0.7804      3269
weighted avg     0.8326    0.8140    0.8203      3269

  Perbandingan vs v1.0 (LR-A-Manual):
    Accuracy  : 79.1300 → 81.4010  ✅↑ +2.2710
    Macro F1  : 0.7582 → 0.7804  ✅↑ +0.0222
    F1 Saran  : 0.5893 → 0.6218  ✅↑ +0.0325
  ✅ LR-v2-A-Manual

HASIL — LR-v2-A-Gabungan
  Accuracy   : 86.16%
  Macro F1   : 0.8212
  F1 Keluhan : 0.9015  Prec:0.9505  Rec:0.8573
  F1 Saran   : 0.6800  Prec:0.5872  Rec:0.8077
  F1 Pujian  :

In [16]:
# Map nama v2.0 → v1.0
V2_TO_V1 = {
    'LR-v2-A-Manual'         : 'LR-A-Manual',
    'LR-v2-A-Gabungan'       : 'LR-A-Gabungan',
    'LR-v2-B-Gabungan'       : 'LR-B-Gabungan',
    'XGB-v2-NoTune-A-Manual'  : 'XGB-NoTune-A-Manual',
    'XGB-v2-NoTune-A-Gabungan': 'XGB-NoTune-A-Gabungan',
    'XGB-v2-NoTune-B-Gabungan': 'XGB-NoTune-B-Gabungan',
    'XGB-v2-Tuned-A-Manual'   : 'XGB-Tuned-A-Manual',
    'XGB-v2-Tuned-A-Gabungan' : 'XGB-Tuned-A-Gabungan',
    'XGB-v2-Tuned-B-Gabungan' : 'XGB-Tuned-B-Gabungan',
}

all_v2 = hasil_lr + hasil_notune + hasil_tuned

print('='*75)
print('PERBANDINGAN LENGKAP v1.0 vs v2.0')
print('='*75)
for h in all_v2:
    v1_key = V2_TO_V1.get(h['nama'])
    if not v1_key or v1_key not in HASIL_V1:
        continue
    v1 = HASIL_V1[v1_key]
    print(f'\n{h["nama"]} vs {v1_key}')
    print(f'{"─"*55}')
    print(f'  {"Metric":12s}  {"v1.0":8s}  {"v2.0":8s}  Delta')
    print(f'  {"─"*50}')
    for label, k_v1, k_v2 in [
        ('Accuracy  ', 'acc',  'acc'),
        ('Macro F1  ', 'f1',   'f1'),
        ('F1 Keluhan', 'f1_k', 'f1_k'),
        ('F1 Saran  ', 'f1_s', 'f1_s'),
        ('F1 Pujian ', 'f1_p', 'f1_p'),
    ]:
        val_v1 = v1[k_v1]
        val_v2 = h[k_v2]*(100 if k_v1=='acc' else 1)
        delta  = val_v2 - val_v1
        ikon   = '✅↑' if delta>0.001 else ('❌↓' if delta<-0.001 else '→')
        print(f'  {label:12s}  {val_v1:.4f}    {val_v2:.4f}    {ikon} {delta:+.4f}')

# Tabel ringkasan akhir
print(f'\n{"="*80}')
print('TABEL RINGKASAN AKHIR')
print(f'{"="*80}')
print(f'{"Model":30s} {"Ver":5s} {"Acc":>8s} {"MacroF1":>8s} {"F1-Kel":>8s} {"F1-Sar":>8s} {"F1-Puj":>8s}')
print(f'{"─"*75}')
# v1.0
for k, v in HASIL_V1.items():
    print(f'{k:30s} v1.0  {v["acc"]:7.2f}%  {v["f1"]:.4f}    {v["f1_k"]:.4f}    {v["f1_s"]:.4f}    {v["f1_p"]:.4f}')
print()
# v2.0
for h in all_v2:
    print(f'{h["nama"]:30s} v2.0  {h["acc"]*100:7.2f}%  {h["f1"]:.4f}    {h["f1_k"]:.4f}    {h["f1_s"]:.4f}    {h["f1_p"]:.4f}')

print(f'\n{"─"*75}')
print('Catatan preprocessing v2.0:')
print('  + Negasi diproteksi dari stopword removal')
print('  + STEM_PROTECT (penangguhan tidak → tangguh)')
print('  + Negation handling (tidak_aktif sebagai token)')
print('  + TF-IDF bigram (1,2)')
print('  + Emoji fix: multi-emoji row di-split ke individual emoji')
print(f'  k_best: Manual={K_MANUAL:,} | Gabungan={K_GABUNGAN:,}')

PERBANDINGAN LENGKAP v1.0 vs v2.0

LR-v2-A-Manual vs LR-A-Manual
───────────────────────────────────────────────────────
  Metric        v1.0      v2.0      Delta
  ──────────────────────────────────────────────────
  Accuracy      79.1300    81.4010    ✅↑ +2.2710
  Macro F1      0.7582    0.7804    ✅↑ +0.0222
  F1 Keluhan    0.8499    0.8615    ✅↑ +0.0116
  F1 Saran      0.5893    0.6218    ✅↑ +0.0325
  F1 Pujian     0.8355    0.8580    ✅↑ +0.0225

LR-v2-A-Gabungan vs LR-A-Gabungan
───────────────────────────────────────────────────────
  Metric        v1.0      v2.0      Delta
  ──────────────────────────────────────────────────
  Accuracy      81.2000    86.1605    ✅↑ +4.9605
  Macro F1      0.7675    0.8212    ✅↑ +0.0537
  F1 Keluhan    0.8684    0.9015    ✅↑ +0.0331
  F1 Saran      0.5941    0.6800    ✅↑ +0.0859
  F1 Pujian     0.8401    0.8821    ✅↑ +0.0420

LR-v2-B-Gabungan vs LR-B-Gabungan
───────────────────────────────────────────────────────
  Metric        v1.0      v2.0   

In [17]:
# ── CELL: Generate Confusion Matrix Semua Model v2.0 ───────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # untuk environment non-GUI
from sklearn.metrics import confusion_matrix
import numpy as np
import os, pickle

CM_DIR = os.path.join(RESULT_DIR, 'confusion_matrix_v2')
os.makedirs(CM_DIR, exist_ok=True)

def plot_confusion_matrix(y_true, y_pred, nama, save_dir):
    """Plot dan simpan confusion matrix sebagai PNG."""
    cm      = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Confusion Matrix — {nama}', fontsize=13, fontweight='bold', y=1.02)

    # Panel kiri: nilai absolut
    im1 = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
    axes[0].set_title('Count', fontsize=11)
    axes[0].set_xticks([0, 1, 2])
    axes[0].set_yticks([0, 1, 2])
    axes[0].set_xticklabels(CLASS_NAMES, rotation=15)
    axes[0].set_yticklabels(CLASS_NAMES)
    axes[0].set_xlabel('Prediksi', fontsize=10)
    axes[0].set_ylabel('Aktual', fontsize=10)
    plt.colorbar(im1, ax=axes[0])
    thresh1 = cm.max() / 2
    for i in range(3):
        for j in range(3):
            axes[0].text(j, i, f'{cm[i,j]:,}',
                ha='center', va='center', fontsize=11,
                color='white' if cm[i,j] > thresh1 else 'black')

    # Panel kanan: persentase (normalized)
    im2 = axes[1].imshow(cm_norm, interpolation='nearest',
                          cmap='Blues', vmin=0, vmax=1)
    axes[1].set_title('Normalized (%)', fontsize=11)
    axes[1].set_xticks([0, 1, 2])
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_xticklabels(CLASS_NAMES, rotation=15)
    axes[1].set_yticklabels(CLASS_NAMES)
    axes[1].set_xlabel('Prediksi', fontsize=10)
    axes[1].set_ylabel('Aktual', fontsize=10)
    plt.colorbar(im2, ax=axes[1])
    for i in range(3):
        for j in range(3):
            axes[1].text(j, i, f'{cm_norm[i,j]:.2%}',
                ha='center', va='center', fontsize=10,
                color='white' if cm_norm[i,j] > 0.5 else 'black')

    plt.tight_layout()
    fname = f'CM_{nama.replace("-","_").replace(" ","_")}.png'
    path  = os.path.join(save_dir, fname)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  ✅ Tersimpan: {fname}')
    return path


def generate_cm_from_pkl(model_key, vec_key, test_file, nama_tampil):
    """Load model dari pkl dan generate confusion matrix."""
    try:
        model = pickle.load(open(
            os.path.join(MODEL_DIR, f'{model_key}.pkl'), 'rb'))
        vec   = pickle.load(open(
            os.path.join(MODEL_DIR, f'vec_{vec_key}.pkl'), 'rb'))

        df_te  = pd.read_csv(os.path.join(SPLIT_DIR, test_file))
        X_te   = df_te['text_v2'].fillna('').values
        y_te   = np.array([LABEL_MAP[l] for l in df_te['label_pks'].values])
        X_test = vec['selector'].transform(vec['tfidf'].transform(X_te))
        y_pred = model.predict(X_test)

        return plot_confusion_matrix(y_te, y_pred, nama_tampil, CM_DIR)
    except FileNotFoundError as e:
        print(f'  ❌ {nama_tampil}: {e}')
        return None


# Daftar semua model v2.0
print('='*60)
print('GENERATE CONFUSION MATRIX — SEMUA MODEL v2.0')
print('='*60)

all_models_cm = [
    # LR
    ('LR_v2_A_Manual',   'LR_v2_A_Manual',
     'v2b_manual_test.csv', 'LR-v2-A-Manual'),
    ('LR_v2_A_Gabungan', 'LR_v2_A_Gabungan',
     'v2b_agab_test.csv',   'LR-v2-A-Gabungan'),
    ('LR_v2_B_Gabungan', 'LR_v2_B_Gabungan',
     'v2b_bgab_test.csv',   'LR-v2-B-Gabungan'),
    # XGB NoTune
    ('XGB_v2_NoTune_A_Manual',   'XGB_v2_NoTune_A_Manual',
     'v2b_manual_test.csv',        'XGB-v2-NoTune-A-Manual'),
    ('XGB_v2_NoTune_A_Gabungan', 'XGB_v2_NoTune_A_Gabungan',
     'v2b_agab_test.csv',          'XGB-v2-NoTune-A-Gabungan'),
    ('XGB_v2_NoTune_B_Gabungan', 'XGB_v2_NoTune_B_Gabungan',
     'v2b_bgab_test.csv',          'XGB-v2-NoTune-B-Gabungan'),
    # XGB Tuned
    ('XGB_v2_Tuned_A_Manual',   'XGB_v2_Tuned_A_Manual',
     'v2b_manual_test.csv',       'XGB-v2-Tuned-A-Manual'),
    ('XGB_v2_Tuned_A_Gabungan', 'XGB_v2_Tuned_A_Gabungan',
     'v2b_agab_test.csv',         'XGB-v2-Tuned-A-Gabungan'),
    ('XGB_v2_Tuned_B_Gabungan', 'XGB_v2_Tuned_B_Gabungan',
     'v2b_bgab_test.csv',         'XGB-v2-Tuned-B-Gabungan'),
]

for model_key, vec_key, test_file, nama in all_models_cm:
    generate_cm_from_pkl(model_key, vec_key, test_file, nama)

print(f'\n✅ Semua confusion matrix tersimpan di:')
print(f'   {CM_DIR}')
print(f'\nFile yang dihasilkan:')
for f in sorted(os.listdir(CM_DIR)):
    print(f'  {f}')

GENERATE CONFUSION MATRIX — SEMUA MODEL v2.0
  ✅ Tersimpan: CM_LR_v2_A_Manual.png
  ✅ Tersimpan: CM_LR_v2_A_Gabungan.png
  ✅ Tersimpan: CM_LR_v2_B_Gabungan.png
  ✅ Tersimpan: CM_XGB_v2_NoTune_A_Manual.png
  ✅ Tersimpan: CM_XGB_v2_NoTune_A_Gabungan.png
  ✅ Tersimpan: CM_XGB_v2_NoTune_B_Gabungan.png
  ✅ Tersimpan: CM_XGB_v2_Tuned_A_Manual.png
  ✅ Tersimpan: CM_XGB_v2_Tuned_A_Gabungan.png
  ✅ Tersimpan: CM_XGB_v2_Tuned_B_Gabungan.png

✅ Semua confusion matrix tersimpan di:
   C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\results\confusion_matrix_v2

File yang dihasilkan:
  CM_LR_v2_A_Gabungan.png
  CM_LR_v2_A_Manual.png
  CM_LR_v2_B_Gabungan.png
  CM_XGB_v2_NoTune_A_Gabungan.png
  CM_XGB_v2_NoTune_A_Manual.png
  CM_XGB_v2_NoTune_B_Gabungan.png
  CM_XGB_v2_Tuned_A_Gabungan.png
  CM_XGB_v2_Tuned_A_Manual.png
  CM_XGB_v2_Tuned_B_Gabungan.png
